# 🤖 Reinforcement Learning — From Zero to PPO
> *A self-contained teaching notebook built from real Colab experiments*

---
**What you will learn:**
- The full RL loop: agent, environment, state, action, reward
- MDPs, Bellman equations, value functions
- Q-Learning → DQN → Policy Gradients → A2C → PPO
- Writing custom Gymnasium environments
- Using Stable-Baselines3 for training & evaluation
- Two real case studies from IoT/sensor-network research

**Prerequisites:** Python, basic PyTorch, basic probability.

## 📋 Table of Contents
1. [Setup & Imports](#setup)
2. [The RL Loop — Interactive Demo](#rl-loop)
3. [MDP Formalism](#mdp)
4. [Value Functions & Bellman Equations](#bellman)
5. [Tabular Q-Learning (from scratch)](#qlearning)
6. [Deep Q-Network (DQN)](#dqn)
7. [Policy Gradients & REINFORCE](#pg)
8. [Actor-Critic (A2C)](#a2c)
9. [PPO — The Gold Standard](#ppo)
10. [Custom Gymnasium Environment](#custom-env)
11. [Case Study 1 — Gateway Selection](#case1)
12. [Case Study 2 — Link Duration Prediction](#case2)
13. [Algorithm Comparison Experiment](#compare)
14. [Exercises](#exercises)

---
## 1. Setup & Imports <a id='setup'></a>

In [ ]:
!pip install -q stable_baselines3 gymnasium matplotlib seaborn tqdm

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium.spaces import Box, Discrete
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.evaluation import evaluate_policy
from copy import deepcopy
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('✅ Setup complete')

---
## 2. The RL Loop — Interactive Demo <a id='rl-loop'></a>

The core RL loop is deceptively simple:

```
while not done:
    action  = agent.choose(state)
    state′, reward, done = environment.step(action)
    agent.learn(state, action, reward, state′)
    state = state′
```

Let's see it with the classic **CartPole** environment (balance a pole on a cart).

In [ ]:
# ── CartPole with a RANDOM agent ──────────────────────────────────────
env = gym.make('CartPole-v1')
obs, info = env.reset(seed=SEED)

print('Observation space:', env.observation_space)  # Box(4,) — cart pos, vel, pole angle, angular vel
print('Action space     :', env.action_space)       # Discrete(2) — push left or right
print('Initial obs      :', obs)

total_reward = 0
step_rewards = []
for step in range(500):
    action = env.action_space.sample()              # random policy
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_rewards.append(total_reward)
    if terminated or truncated:
        print(f'  Episode ended at step {step+1}, total reward = {total_reward}')
        obs, info = env.reset()
        total_reward = 0
env.close()
print('\n📌 Key takeaway: a random policy fails fast. We need to LEARN.')

---
## 3. MDP Formalism <a id='mdp'></a>

A **Markov Decision Process** is the mathematical backbone of RL.

$$\mathcal{M} = (\mathcal{S},\ \mathcal{A},\ \mathcal{P},\ \mathcal{R},\ \gamma)$$

| Symbol | Name | Example (CartPole) |
|---|---|---|
| $\mathcal{S}$ | State space | 4 continuous values |
| $\mathcal{A}$ | Action space | {0=left, 1=right} |
| $\mathcal{P}(s'\|s,a)$ | Transition probability | Physics simulator |
| $\mathcal{R}(s,a,s')$ | Reward | +1 per timestep alive |
| $\gamma$ | Discount factor | 0.99 |

### The Markov Property
> The future depends **only on the present state**, not on history.

$$P(s_{t+1} \mid s_t, a_t) = P(s_{t+1} \mid s_0, a_0, \ldots, s_t, a_t)$$

### Discount factor $\gamma$
Controls the trade-off between immediate and future rewards:

In [ ]:
# Visualise effect of gamma on how much future reward matters
steps = np.arange(0, 30)
fig, ax = plt.subplots(figsize=(10, 4))
for gamma, color in [(0.5, 'red'), (0.9, 'orange'), (0.99, 'green'), (1.0, 'blue')]:
    weights = gamma ** steps
    ax.plot(steps, weights, label=f'γ={gamma}', color=color)
ax.set_xlabel('Steps into the future')
ax.set_ylabel('Discount weight γᵏ')
ax.set_title('How much do we value rewards k steps in the future?')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('γ≈0: myopic agent   |   γ≈1: far-sighted agent')

---
## 4. Value Functions & Bellman Equations <a id='bellman'></a>

### Discounted Return
$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots = \sum_{k=0}^\infty \gamma^k r_{t+k+1}$$

### State-Value Function
$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t = s]$$

### Action-Value Function (Q-function)
$$Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid s_t=s,\ a_t=a]$$

### Advantage Function
$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$
*How much better is action $a$ vs the average action in state $s$?*

### Bellman Optimality Equation
$$Q^*(s,a) = \mathbb{E}_{s'}\left[r + \gamma \max_{a'} Q^*(s',a')\right]$$

This is the key identity that Q-Learning exploits directly.

In [ ]:
# Concrete example: compute returns for a sample trajectory
rewards = [1, 1, 1, 1, 0]      # 4 steps alive, then falls
gamma   = 0.99

# G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
def compute_returns(rewards, gamma):
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

returns = compute_returns(rewards, gamma)
print(f'Rewards : {rewards}')
print(f'Returns : {[round(g,4) for g in returns]}')
print(f'\nG_0 = {returns[0]:.4f} — total discounted value from t=0')
print(f'G_3 = {returns[3]:.4f} — only 1 reward left, discounted once')

---
## 5. Tabular Q-Learning (from scratch) <a id='qlearning'></a>

For small, discrete state & action spaces we can store $Q(s,a)$ in a table and update it with:
$$Q(s,a) \leftarrow Q(s,a) + \alpha\underbrace{\left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]}_{\text{TD error}}$$

We use **FrozenLake** — a 4×4 grid where the agent must reach the goal without falling into holes.

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=False)
n_states  = env.observation_space.n   # 16
n_actions = env.action_space.n        # 4

# ── Hyperparameters ───────────────────────────────────────────────────
ALPHA      = 0.8     # learning rate
GAMMA      = 0.95    # discount
EPSILON    = 1.0     # exploration (decays)
EPS_DECAY  = 0.995
EPS_MIN    = 0.01
N_EPISODES = 2000

Q = np.zeros((n_states, n_actions))
episode_rewards = []

for ep in range(N_EPISODES):
    state, _ = env.reset(seed=ep)
    total_r   = 0
    for _ in range(100):
        # ε-greedy action selection
        if random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = env.step(action)

        # Bellman update
        td_target = reward + GAMMA * np.max(Q[next_state]) * (not terminated)
        td_error  = td_target - Q[state, action]
        Q[state, action] += ALPHA * td_error

        state   = next_state
        total_r += reward
        if terminated or truncated:
            break

    episode_rewards.append(total_r)
    EPSILON = max(EPS_MIN, EPSILON * EPS_DECAY)

# ── Plot learning curve ───────────────────────────────────────────────
window = 100
smoothed = pd.Series(episode_rewards).rolling(window).mean()
plt.figure(figsize=(10, 3))
plt.plot(smoothed, color='steelblue')
plt.xlabel('Episode'); plt.ylabel(f'Avg reward ({window}-ep window)')
plt.title('Q-Learning on FrozenLake (non-slippery)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f'Final success rate (last 100 eps): {np.mean(episode_rewards[-100:]):.1%}')

In [ ]:
# Visualise the learned Q-table
action_names = ['←', '↓', '→', '↑']
best_actions = np.argmax(Q, axis=1).reshape(4, 4)
best_q       = np.max(Q, axis=1).reshape(4, 4)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(best_q, annot=True, fmt='.2f', cmap='YlGn', ax=axes[0])
axes[0].set_title('Max Q-value per state (= V* estimate)')

annot = [[action_names[best_actions[r,c]] for c in range(4)] for r in range(4)]
sns.heatmap(best_q, annot=annot, fmt='', cmap='YlGn', ax=axes[1])
axes[1].set_title('Greedy policy (best action per state)')
plt.tight_layout(); plt.show()

---
## 6. Deep Q-Network (DQN) <a id='dqn'></a>

Q-tables break for large/continuous state spaces. **DQN** replaces the table with a neural network.

$$Q(s, a;\ \theta) \approx Q^*(s, a)$$

**Two key stabilisation tricks:**
1. **Experience Replay** — store $(s, a, r, s')$ in a buffer; sample random mini-batches  
   → breaks temporal correlations between updates
2. **Target Network** $\theta^-$ — a frozen copy updated slowly  
   → prevents the target chasing itself

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[\left(r + \gamma \max_{a'} Q(s', a';\ \theta^-) - Q(s, a;\ \theta)\right)^2\right]$$

⚠️ **DQN only supports DISCRETE action spaces.**

In [ ]:
# DQN on CartPole using Stable-Baselines3
env = gym.make('CartPole-v1')

dqn_agent = DQN(
    'MlpPolicy', env,
    learning_rate   = 1e-3,
    buffer_size     = 10_000,
    learning_starts = 500,
    batch_size      = 64,
    gamma           = 0.99,
    target_update_interval = 500,
    exploration_fraction   = 0.3,
    exploration_final_eps  = 0.02,
    verbose = 0,
    seed    = SEED
)

print('Training DQN on CartPole...')
dqn_agent.learn(total_timesteps=30_000)
mean_r, std_r = evaluate_policy(dqn_agent, env, n_eval_episodes=20)
print(f'DQN  → Mean reward: {mean_r:.1f} ± {std_r:.1f}  (max possible ≈ 500)')
env.close()

---
## 7. Policy Gradients & REINFORCE <a id='pg'></a>

Instead of learning a value function, directly optimise the policy parameters $\theta$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[G_t \cdot \nabla_\theta \log \pi_\theta(a_t \mid s_t)\right]$$

**Intuition:** 
- If the trajectory got **high return** → push up the log-probability of those actions  
- If the trajectory got **low return** → push down those probabilities

**REINFORCE** uses Monte Carlo returns directly (full episode).

**Problem:** Very high variance → slow convergence. Baseline subtraction (e.g. $G_t - b$) helps.

In [ ]:
# Minimal REINFORCE implementation (no SB3)
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, act_dim), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.net(x)

env = gym.make('CartPole-v1')
policy = PolicyNet(obs_dim=4, act_dim=2)
optimiser = torch.optim.Adam(policy.parameters(), lr=1e-2)

episode_returns = []
for episode in range(400):
    obs, _ = env.reset()
    log_probs, rewards = [], []

    for _ in range(500):
        obs_t  = torch.FloatTensor(obs)
        probs  = policy(obs_t)
        dist   = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))

        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        if terminated or truncated: break

    # Compute discounted returns
    returns = compute_returns(rewards, gamma=0.99)
    returns = torch.FloatTensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalise

    # Policy gradient loss: -E[G_t * log π(a|s)]
    loss = -torch.stack(log_probs) * returns
    loss = loss.sum()

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    episode_returns.append(sum(rewards))

env.close()
smoothed = pd.Series(episode_returns).rolling(30).mean()
plt.figure(figsize=(10, 3))
plt.plot(episode_returns, alpha=0.3, color='grey')
plt.plot(smoothed, color='darkorange', linewidth=2, label='30-ep avg')
plt.xlabel('Episode'); plt.ylabel('Total reward')
plt.title('REINFORCE on CartPole'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 8. Actor-Critic (A2C) <a id='a2c'></a>

REINFORCE has high variance because $G_t$ depends on the entire future trajectory.  
**Actor-Critic** solves this by using a learned **baseline** (the Critic).

```
┌──────────────┐         ┌──────────────┐
│  ACTOR  π_θ  │         │  CRITIC V_φ  │
│              │         │              │
│  s → π(a|s) │         │  s  → V(s)  │
│  (policy)    │         │  (evaluator) │
└──────────────┘         └──────────────┘
```

**Advantage estimate:**
$$\hat{A}_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t) \quad \text{(TD error)}$$

**Actor loss:**  $\mathcal{L}_\text{actor} = -\hat{A}_t \cdot \log \pi_\theta(a_t|s_t)$

**Critic loss:**  $\mathcal{L}_\text{critic} = \hat{A}_t^2$

**Total loss:**  $\mathcal{L} = \mathcal{L}_\text{actor} + c_1 \mathcal{L}_\text{critic} - c_2 \mathcal{H}[\pi_\theta]$

In [ ]:
env = gym.make('CartPole-v1')

a2c_agent = A2C(
    'MlpPolicy', env,
    learning_rate = 7e-4,
    gamma         = 0.99,
    n_steps       = 5,
    verbose       = 0,
    seed          = SEED
)
print('Training A2C...')
a2c_agent.learn(total_timesteps=30_000)
mean_r, std_r = evaluate_policy(a2c_agent, env, n_eval_episodes=20)
print(f'A2C  → Mean reward: {mean_r:.1f} ± {std_r:.1f}')
env.close()

---
## 9. PPO — The Gold Standard <a id='ppo'></a>

**The problem with vanilla policy gradient:**  
A bad update can collapse the policy. Making the learning rate small enough to be safe makes training slow.

**PPO's solution:** Clip the update ratio to stay close to the old policy.

Define the probability ratio:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

**PPO-Clip objective:**
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) A_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t\right)\right]$$

With $\varepsilon = 0.2$ (default), the ratio is forced to stay in $[0.8, 1.2]$.  
This prevents the policy from changing too drastically in one step.

### Why PPO works so well:
- Works for discrete AND continuous actions  
- Stable, predictable training  
- No replay buffer needed  
- Scales well with parallel environments

In [ ]:
# Visualise the PPO clipping mechanism
eps   = 0.2
ratios = np.linspace(0.4, 1.6, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, A, title in zip(axes, [1.0, -1.0], ['Positive advantage A>0', 'Negative advantage A<0']):
    unclipped = ratios * A
    clipped   = np.clip(ratios, 1 - eps, 1 + eps) * A
    objective = np.minimum(unclipped, clipped)

    ax.plot(ratios, unclipped, '--', color='grey',   label='Unclipped: r·A')
    ax.plot(ratios, clipped,   '--', color='orange', label='Clipped: clip(r,0.8,1.2)·A')
    ax.plot(ratios, objective, '-',  color='steelblue', linewidth=2.5, label='PPO objective (min)')
    ax.axvline(1-eps, color='red', linestyle=':', alpha=0.5)
    ax.axvline(1+eps, color='red', linestyle=':', alpha=0.5, label='Clip bounds')
    ax.axvline(1.0,   color='black', linestyle='-', alpha=0.3, label='r=1 (no change)')
    ax.set_xlabel('Ratio r_t = π_new / π_old')
    ax.set_ylabel('Objective value')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('PPO Clipping: prevents overly large policy updates', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
env = gym.make('CartPole-v1')

ppo_agent = PPO(
    'MlpPolicy', env,
    learning_rate = 3e-4,
    n_steps       = 1024,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    clip_range    = 0.2,
    verbose       = 0,
    seed          = SEED
)
print('Training PPO...')
ppo_agent.learn(total_timesteps=30_000)
mean_r, std_r = evaluate_policy(ppo_agent, env, n_eval_episodes=20)
print(f'PPO  → Mean reward: {mean_r:.1f} ± {std_r:.1f}')
env.close()

---
## 10. Custom Gymnasium Environment <a id='custom-env'></a>

Every Gymnasium environment must implement 4 methods:

| Method | Returns | Purpose |
|---|---|---|
| `__init__` | — | Define `observation_space`, `action_space` |
| `reset()` | `(obs, info)` | Start new episode |
| `step(action)` | `(obs, reward, terminated, truncated, info)` | One timestep |
| `render()` | — | Optional visualisation |

### Template

In [ ]:
class TemplateEnv(gym.Env):
    """Minimal custom env template."""

    def __init__(self, max_steps=100):
        super().__init__()
        self.max_steps = max_steps
        self.step_count = 0

        # ── REQUIRED: define spaces ───────────────────────────────────
        self.observation_space = Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = Discrete(3)  # 3 discrete actions

    def reset(self, seed=None):
        super().reset(seed=seed)         # sets self.np_random
        self.step_count = 0
        obs = self.observation_space.sample()  # random start
        return obs.astype(np.float32), {}

    def step(self, action):
        self.step_count += 1

        # TODO: implement your dynamics here
        obs    = self.observation_space.sample().astype(np.float32)
        reward = float(action == 0)       # reward for choosing action 0
        terminated = False
        truncated  = self.step_count >= self.max_steps
        info = {'step': self.step_count}

        return obs, reward, terminated, truncated, info

# ── Sanity check ─────────────────────────────────────────────────────
env = TemplateEnv()
obs, info = env.reset()
print('obs shape:', obs.shape, '| dtype:', obs.dtype)
for _ in range(3):
    a = env.action_space.sample()
    obs, r, term, trunc, info = env.step(a)
    print(f'  action={a}  reward={r}  terminated={term}  truncated={trunc}')
print('✅ Environment passes basic sanity check')

---
## 11. Case Study 1 — Gateway Selection in IoT Networks <a id='case1'></a>

**Real problem from research:** A set of $N$ mobile gateways (drones) move in 3D space.  
At each step, an agent selects **which gateway to route traffic through**, optimising:
1. **Channel Quality Index (CQI)** — prefer gateways closest to base stations
2. **Load Fairness (Jain's Index)** — don't overload any one gateway

$$r = \frac{1}{2}\left(\frac{\text{CQI}_{\text{chosen}}}{\max_i \text{CQI}_i} + \frac{(\sum_i L_i)^2}{N \sum_i L_i^2}\right)$$

The second term is **Jain's Fairness Index** — equals 1 when all loads are equal.

In [ ]:
class GatewayEnv(gym.Env):
    """
    Gateway selection in a 3D sensor network.
    State : positions (x,y,z) of N gateways + 2 base stations,
            CQI values, load values.
    Action: discrete — which gateway to select.
    Reward: combination of CQI reward and Jain's fairness index.
    """
    def __init__(self, n_gateways=5, max_speed=10, choice_load=3):
        super().__init__()
        self.ng          = n_gateways
        self.maxspeed    = max_speed
        self.choice_load = choice_load
        self.max_dist    = np.sqrt(30000)
        self.end_step    = 1
        self.timestep    = 0

        obs_size = 3 * (self.ng + 2) + 2 * self.ng
        self.observation_space = Box(low=0, high=100, shape=(obs_size,), dtype=np.uint8)
        self.action_space = Discrete(self.ng)
        self._init_positions()

    def _init_positions(self):
        self.x1, self.y1, self.z1 = [random.randint(0,100) for _ in range(3)]
        self.x2, self.y2, self.z2 = [random.randint(0,100) for _ in range(3)]
        self.px = [np.random.randint(0,100) for _ in range(self.ng)] + [self.x1, self.x2]
        self.py = [np.random.randint(0,100) for _ in range(self.ng)] + [self.y1, self.y2]
        self.pz = [np.random.randint(0,100) for _ in range(self.ng)] + [self.z1, self.z2]
        self.loads = [np.random.randint(1,10) for _ in range(self.ng)]
        self._update_cqis()

    def _dist(self, i):
        d1 = np.sqrt((self.px[i]-self.x1)**2+(self.py[i]-self.y1)**2+(self.pz[i]-self.z1)**2)
        d2 = np.sqrt((self.px[i]-self.x2)**2+(self.py[i]-self.y2)**2+(self.pz[i]-self.z2)**2)
        return min(d1, d2)

    def _update_cqis(self):
        self.cqis = [-(15/self.max_dist)*self._dist(i)+15 for i in range(self.ng)]

    def get_obs(self):
        return np.array(self.px + self.py + self.pz + self.cqis + self.loads, dtype=np.uint8)

    def reset(self, seed=None):
        super().reset(seed=seed)
        self._init_positions()
        self.timestep = 0
        return self.get_obs(), {}

    def step(self, action):
        self.timestep += 1
        mycqi = self.cqis[action]
        self.loads[action] += self.choice_load

        cqi_reward   = mycqi / max(self.cqis)
        sq_sm = sum(self.loads)**2
        sm_sq = sum(x**2 for x in self.loads)
        load_reward  = sq_sm / (self.ng * sm_sq)
        reward = (cqi_reward + load_reward) / 2

        for i in range(self.ng):
            for coord, p in enumerate([self.px, self.py, self.pz]):
                delta = random.randint(-self.maxspeed, self.maxspeed)
                new_v = p[i] + delta
                if 0 < new_v < 100:
                    p[i] = new_v
        self._update_cqis()

        terminated = self.timestep >= self.end_step
        return self.get_obs(), reward/self.end_step, terminated, terminated, \
               {'cqi_reward': cqi_reward, 'fairness': load_reward}

# ── Quick test ────────────────────────────────────────────────────────
env = GatewayEnv(n_gateways=5)
obs, _ = env.reset()
print('Observation shape:', obs.shape)
print('Action space     :', env.action_space)
obs, r, term, trunc, info = env.step(0)
print(f'Step reward: {r:.4f}  CQI reward: {info["cqi_reward"]:.4f}  Fairness: {info["fairness"]:.4f}')

In [ ]:
# Train PPO on the gateway environment
N_GATEWAYS         = 5
TRAINING_STEPS     = 10_000
N_EVAL_INFERENCE   = 20
LR                 = 0.01

env = GatewayEnv(n_gateways=N_GATEWAYS)
policy_kwargs = dict(activation_fn=torch.nn.ReLU, net_arch=[128, 64, 32, 16])

results = {}
for agent_name, AgentClass in [('PPO', PPO), ('A2C', A2C), ('DQN', DQN)]:
    print(f'Training {agent_name}...')
    agent = AgentClass(
        'MlpPolicy', env,
        learning_rate = LR,
        policy_kwargs = policy_kwargs,
        verbose       = 0,
        seed          = SEED
    )
    agent.learn(total_timesteps=TRAINING_STEPS)
    mean_r, std_r = evaluate_policy(agent, env, n_eval_episodes=N_EVAL_INFERENCE)
    results[agent_name] = (agent, mean_r, std_r)
    print(f'  {agent_name}: {mean_r:.4f} ± {std_r:.4f}')

print('\n✅ Training complete')

In [ ]:
# Evaluate trained agents over N_EVAL_INFERENCE steps
def run_inference(agent, env, steps=20):
    env.end_step = steps
    obs, _ = env.reset()
    cqi_list, load_list, fairness_list, reward_list = [], [], [], []
    for _ in range(steps):
        cqi_list.append(deepcopy(env.cqis))
        load_list.append(deepcopy(env.loads))
        action, _ = agent.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(action)
        r *= steps
        fairness_list.append(info['fairness'])
        reward_list.append(r)
        if term or trunc: break
    return np.array(cqi_list), np.array(load_list), np.array(fairness_list), np.array(reward_list)

env = GatewayEnv(n_gateways=N_GATEWAYS)
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

colors = {'PPO': 'steelblue', 'A2C': 'darkorange', 'DQN': 'green'}
for name, (agent, mr, sr) in results.items():
    _, load_arr, fairness_arr, reward_arr = run_inference(agent, env, steps=50)
    axes[0].plot(fairness_arr, label=f'{name} (mean={mr:.3f})', color=colors[name])
    axes[1].plot(reward_arr,   label=name, color=colors[name])
    axes[2].plot(load_arr.sum(axis=1), label=f'{name} total load', color=colors[name])

axes[0].set_title("Jain's Fairness Index over time");   axes[0].set_ylabel('Fairness')
axes[1].set_title('Total reward per timestep');          axes[1].set_ylabel('Reward')
axes[2].set_title('Cumulative load across all gateways');axes[2].set_ylabel('Total load')
for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlabel('Timestep')
plt.suptitle('Gateway Selection — Agent Comparison', fontsize=14)
plt.tight_layout(); plt.show()

---
## 12. Case Study 2 — Link Duration Prediction <a id='case2'></a>

**Problem:** A moving gateway follows a known trajectory. A stationary sensor has radio range $R$.  
The agent sees the last 3 positions and must predict **how many timesteps** the link will remain active.

**Key difference from Case 1:**  
- Action space is **continuous** (`Box`) → DQN cannot be used
- Reward is MAPE-style: $r = -|a - d_{\text{true}}| / d_{\text{true}}$

In [ ]:
class LinkDurationEnv(gym.Env):
    """
    Predict remaining link duration for a gateway moving in 3D.
    Action: continuous — predicted number of steps until link drops.
    Obs   : sliding window of last 3 positions (shape 3×3).
    Reward: negative MAPE.
    """
    def __init__(self, max_steps=1000, radius=10):
        super().__init__()
        self.max_steps = max_steps
        self.radius    = radius
        self.sensor    = np.array([25, 25, 25])

        # Continuous action: predicted duration
        self.action_space = Box(low=0, high=max_steps, shape=(1,), dtype=np.float32)
        # Observation: 3 last positions (3×3 matrix flattened)
        self.observation_space = Box(low=0, high=100, shape=(3, 3), dtype=np.float32)

        self.trajectory = [(0.8*t + 16, 0.8*t + 16, 0.8*t + 16) for t in range(max_steps)]
        self.true_duration = self._compute_duration()
        self.current_step  = 0

    def _compute_duration(self):
        t = 0
        for pos in self.trajectory:
            if np.linalg.norm(np.array(pos) - self.sensor) <= self.radius:
                t += 1
            elif t > 0:
                break
        return max(t, 1)

    def _get_obs(self):
        t = self.current_step
        buf = []
        for offset in [2, 1, 0]:
            idx = max(0, t - offset)
            buf.append(list(self.trajectory[idx]))
        return np.array(buf, dtype=np.float32)

    def reset(self, seed=None):
        self.current_step = 0
        return self._get_obs(), {}

    def step(self, action):
        prediction = float(action[0])
        reward     = -abs(prediction - self.true_duration) / self.true_duration
        self.current_step += 1
        done = self.current_step >= len(self.trajectory)
        obs  = np.zeros((3, 3), dtype=np.float32) if done else self._get_obs()
        return obs, reward, done, done, {'true_duration': self.true_duration, 'predicted': prediction}

env = LinkDurationEnv()
print(f'True link duration: {env.true_duration} steps')
print(f'Action space: {env.action_space}')
print(f'Obs space:    {env.observation_space}')

In [ ]:
# ⚠️ Only PPO and A2C support continuous (Box) action spaces
# DQN would raise: AssertionError — only supports Discrete

from stable_baselines3.common.env_util import make_vec_env

env = LinkDurationEnv()
vec_env = make_vec_env(lambda: LinkDurationEnv(), n_envs=1)

policy_kwargs = dict(activation_fn=torch.nn.ReLU, net_arch=[128, 64, 32, 16])
agent = A2C('MlpPolicy', vec_env, learning_rate=1e-4, policy_kwargs=policy_kwargs, verbose=0)

print('Training A2C on Link Duration...')
agent.learn(total_timesteps=50_000)

# Test
obs, _ = env.reset()
preds = []
for _ in range(50):
    action, _ = agent.predict(obs.reshape(1, 3, 3), deterministic=True)
    obs, r, term, trunc, info = env.step(action[0])
    preds.append(info['predicted'])
    if term or trunc: break

print(f'True duration : {env.true_duration}')
print(f'Mean predicted: {np.mean(preds):.2f}  (ideal = {env.true_duration})')
print(f'MAPE          : {abs(np.mean(preds) - env.true_duration)/env.true_duration:.1%}')

---
## 13. Algorithm Comparison Experiment <a id='compare'></a>

Let's compare all three algorithms on CartPole to clearly see their differences.

In [ ]:
env    = gym.make('CartPole-v1')
STEPS  = 30_000
N_EVAL = 30

algorithms = {
    'DQN': DQN('MlpPolicy', env, learning_rate=1e-3, verbose=0, seed=SEED),
    'A2C': A2C('MlpPolicy', env, learning_rate=7e-4, verbose=0, seed=SEED),
    'PPO': PPO('MlpPolicy', env, learning_rate=3e-4, verbose=0, seed=SEED),
}

comparison = {}
for name, agent in algorithms.items():
    print(f'Training {name}...')
    agent.learn(total_timesteps=STEPS)
    mean_r, std_r = evaluate_policy(agent, env, n_eval_episodes=N_EVAL)
    comparison[name] = (mean_r, std_r)
    print(f'  {name:4s}: {mean_r:6.1f} ± {std_r:.1f}')

env.close()

# ── Bar chart ─────────────────────────────────────────────────────────
names  = list(comparison.keys())
means  = [comparison[n][0] for n in names]
stds   = [comparison[n][1] for n in names]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, means, yerr=stds, capsize=8,
              color=['green', 'darkorange', 'steelblue'], alpha=0.85)
ax.set_ylabel('Mean reward (30 episodes)')
ax.set_title(f'CartPole-v1 — {STEPS:,} training steps')
ax.axhline(500, color='red', linestyle='--', alpha=0.5, label='Max possible (500)')
ax.legend()
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{mean:.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Summary Table

| Property | DQN | A2C | PPO |
|---|---|---|---|
| **Action space** | Discrete only | Discrete + Continuous | Discrete + Continuous |
| **Policy type** | Off-policy | On-policy | On-policy |
| **Sample efficiency** | Higher (replay) | Lower | Medium |
| **Stability** | Medium | Medium | High (clipping) |
| **Training speed** | Slower | Fast | Medium |
| **GPU benefit** | Low (MLP) | Low (MLP) | Low (MLP) |
| **Use when** | Discrete, off-policy preferred | Fast prototyping | Default choice |

> **Rule of thumb:** Start with PPO. If discrete and sample efficiency matters, try DQN. If speed matters most, try A2C.

---
## 14. Exercises <a id='exercises'></a>

### 🟢 Beginner
1. Change the discount factor $\gamma$ in Q-Learning from 0.95 to 0.5. How does the learned policy change? Why?
2. In the CartPole REINFORCE implementation, remove the return normalisation. What happens to training stability?
3. Run DQN on `LunarLander-v3`. How many timesteps until it consistently lands?

### 🟡 Intermediate
4. Modify `GatewayEnv` to add a **third base station**. Does the agent need more training steps?
5. Add a **curriculum**: start the gateway env with `end_step=1`, gradually increase to 10. Does this help convergence?
6. Replace Jain's Fairness Index in the reward with a **max-load penalty**: $r_{\text{load}} = -\max_i L_i / \sum_i L_i$. Compare results.
7. Try `VecNormalize` wrapper from SB3 on `LinkDurationEnv`. Does reward normalisation help?

### 🔴 Advanced
8. Implement **Prioritised Experience Replay** for the DQN. Does it improve on CartPole?
9. Add **multiple sensors** to `LinkDurationEnv` and predict the duration for each simultaneously (multi-output regression). What architecture works?
10. Train an agent on `GatewayEnv` with PPO and **export it** (`agent.save()`). Load it and run inference without Stable-Baselines3 by extracting the underlying PyTorch model weights.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           RL Teaching Notebook — Complete!                   ║
╠══════════════════════════════════════════════════════════════╣
║  Covered:                                                    ║
║   ✅ MDP, Bellman equations, value functions                 ║
║   ✅ Q-Learning (tabular, FrozenLake)                        ║
║   ✅ DQN (CartPole, SB3)                                     ║
║   ✅ REINFORCE (from scratch, PyTorch)                       ║
║   ✅ A2C (SB3)                                               ║
║   ✅ PPO — theory + clipping visualisation                   ║
║   ✅ Custom Gymnasium env template                           ║
║   ✅ Case study: Gateway selection (discrete, fairness)      ║
║   ✅ Case study: Link duration prediction (continuous)       ║
║   ✅ Algorithm comparison experiment                         ║
╚══════════════════════════════════════════════════════════════╝
""")